# Node 05b · V3 生产 MD（99 条轨迹） · v0.2.2

物理协议**逐字转录自 V2 `run_md2.sh`** 的 `write_mdp_*` 函数（V2 从不把 MDP 写成独立文件，
它们在运行时由 bash 函数就地生成）。转录已通过逐参数比对验证：em 9 项、nvt 27 项、
nvt_continue 25 项、npt 30 项、md 29 项，**除下列四处声明偏离外无任何未声明差异**。

## 四处受控偏离及其理由

| # | V2 | V3 | 理由 |
|---|---|---|---|
| a | `gen_seed = -1` | `r1/r2/r3 = 42/43/44`，写入 manifest | −1 由时钟/PID 取种，replica 既不可复现也未被记录，与 σ_PBSA 的方差论证相抵触 |
| b | `define = -DPOSRES` | `-DPOSRES -DPOSRESLIG` | V2 只约束蛋白，配体在 NVT/NPT 期间自由，削弱"轨迹起自所选对接姿势"的表述；`posre_ligand.itp` 已由 ACPYPE 产出 |
| c | `tc-grps = Protein_UNL …` | `Protein_MOL …` | 残基名由 ACPYPE 决定：V2 = `UNL`，V3 = `MOL`。非风格改动，是事实 |
| d | 硬编码组号（`-cg 1 13`、mindist `16`/`19`） | 一律**按名解析** | V3 的 index 重新构建，序号不保证一致 |
| e | 平衡路径由"条件闸"决定 | `EQUILIBRATION_MODE = "staged"` **显式声明** | V2 的闸恒真（见下），"主路径"是死代码；协议不应取决于一次组集合运算的偶然 |

## 关于 (e)：V2 的 QC 闸恒真，磁盘可证

V2 比较的是 **System 对 SOL**，而 SOL ⊆ System，故最近的原子对即某个水分子**自身的 O–H 键**。
TIP3P 平衡 O–H = 0.09572 nm，EM 段无 constraints，实测 **0.0944 nm** —— 对任何溶剂化体系
都必然 < 0.12 nm 阈值。磁盘证据（V2 全部 138 条溶剂化轨迹）：

| | 计数 |
|---|---|
| `03_min/em_strong.gro`（分段路径） | **138** |
| `04_nvt/nvt300.gro`（分段路径） | **138** |
| `04_nvt/nvt.gro`（主路径） | **0** |

故 V3 声明分段平衡为唯一路径 —— 与 V2 既成行为逐字一致。原闸仍照算并写入 `qc.json`
作为诊断（`legacy_gate_branch`），若将来某条给出 `main`，那是需要解释的信号。

## 真实平衡协议（Methods 必须如实写出）

| 阶段 | 参数 |
|---|---|
| EM 1 | 最速下降，`emtol 500`，≤ 50,000 步 |
| EM 2 | 最速下降，`emtol 100`，≤ 200,000 步，起自 EM 1 输出 |
| NVT 1 | **50 K**，20 ps，`gen_vel = yes`，位置限制 |
| NVT 2 | 300 K，80 ps，续算，位置限制 |
| NPT | 300 K，100 ps，C-rescale，`refcoord_scaling = com`，限制 |
| MD | 300 K，100 ns，Parrinello-Rahman，无限制 |

NVT 总时长仍为 100 ps；但**两段式 EM 与 50 K 预平衡此前从未被记载**，
且 50 K 起步是一种有意为之的缓和升温协议，不可略去。

**其余全部保持 V2 原样**：dt 2 fs · LINCS h-bonds(iter 2/order 4) · Verlet · nstlist 20 ·
PME · rc = rvdw = 1.0 nm · V-rescale（τ 0.2 平衡 / 0.5 生产）· NPT 用 **C-rescale +
`refcoord_scaling = com`** · 生产用 Parrinello-Rahman · `nstxout-compressed = 5000` ·
EM 500/50000 · NVT 100 ps · NPT 100 ps · 生产 100 ns。

## 已核对的算术
50,000,000 步 × 2 fs = 100 ns；每 5,000 步一帧 = 10 ps → **10,001 帧**（phase2/3/4 的假定）；
MM-PBSA 取 5001–10000 间隔 25 = 200 帧 = 50.01–99.76 ns（2026-10-31 更正：原记 interval 10 / 500 帧）；phase4 取 5000 起 stride 10 =
501 帧 = 50.00–99.99 ns。两者同窗口、差一帧，Methods 须一次说清。

## QC 诊断仍逐轨迹记录
每条轨迹写 `qc.json`：`fmax` / `mindist_nm` / `equilibration`（声明值）/
`legacy_gate_branch`（V2 原闸的判定）/ `legacy_gate_agrees`。Cell 6 断言 99 条路径统一。

## 前置
Node 05 制备已冻结：33 体系、六道不变量全过、`Node05_prep_freeze.json`
sha256 `c81f4fa0ddac…`。本 notebook 只**新增** `md_solv_v3/<系统>/r{1,2,3}/`，
不改动任何制备产物（`complex.gro` / `topol.top` 只读复制到各 replica 目录）。


In [3]:
# Cell 1 · 配置 + 前置核验
from pathlib import Path
import sys, json, importlib
PR = Path("/mnt/hpc/home/wanghongmei/lang2").resolve()
if str(PR) not in sys.path: sys.path.insert(0, str(PR))
PY    = "/mnt/hpc/home/wanghongmei/anaconda3/envs/md_env/bin/python"
GMXRC = "/mnt/hpc/home/wanghongmei/apps/gromacs-2025.4-cuda12.1/bin/GMXRC"

TARGET   = "6W63"
OUT_ROOT = PR/"md_solv_v3"
REPLICAS = ["r1", "r2", "r3"]

from core import node05_md_prep as N5; importlib.reload(N5)
assert N5.NODE05_VERSION >= "0.2.3", f"需 v0.2.3（声明式平衡路径），当前 {N5.NODE05_VERSION}"

freeze = json.loads((OUT_ROOT/"Node05_prep_freeze.json").read_text())
sysdirs = [OUT_ROOT/f"{s['job']['name']}__{TARGET}__{s['job']['pose_id']}" for s in freeze["systems"]]
assert len(sysdirs) == 33 and all(s["all_passed"] for s in freeze["systems"])

seeds = N5.assert_replica_seeds_distinct(REPLICAS)
print("node05_md_prep", N5.NODE05_VERSION)
print(f"制备冻结: {freeze['n_systems']} 体系 / 六道不变量全过 | 受体 protein.gro "
      f"{freeze['receptor']['protein.gro']['sha256_excl_title'][:16]}…")
print("replica 种子（显式、互异、将写入 manifest）:", seeds)
print("配体残基名:", N5.detect_ligand_resname(str(sysdirs[0]/"ligand.acpype"/"ligand_GMX.itp")),
      " (V2 为 UNL —— 下游 phase2/3/4 的选择串须同步)")
print("平衡路径（声明）:", N5.EQUILIBRATION_MODE, "— V2 138/138 实测同此")
print(f"计划轨迹: {len(sysdirs)} × {len(REPLICAS)} = {len(sysdirs)*len(REPLICAS)}")

node05_md_prep 0.2.3
制备冻结: 33 体系 / 六道不变量全过 | 受体 protein.gro 61dc910cf9902598…
replica 种子（显式、互异、将写入 manifest）: {'r1': 42, 'r2': 43, 'r3': 44}
配体残基名: MOL  (V2 为 UNL —— 下游 phase2/3/4 的选择串须同步)
平衡路径（声明）: staged — V2 138/138 实测同此
计划轨迹: 33 × 3 = 99


In [4]:
# Cell 2 · I5（第七道断言）+ 为 topol.top 补配体位置限制
# I5：ligand_GMX.gro 出自 ACPYPE 内部 OpenBabel 的写出路径，其原子顺序若与 itp 不同，
#     配体会带着被打乱的几何进入 MD，而 I1–I4 全都察觉不到。这是 I2b 未触及的一跳。
i5, patched = {}, {"patched": 0, "already": 0}
for d in sysdirs:
    itp = d/"ligand.acpype"/"ligand_GMX.itp"
    i5[d.name] = N5.assert_ligand_coords_match_itp(str(d/"ligand.sdf"),
                                                   str(d/"ligand.acpype"/"ligand_GMX.gro"),
                                                   str(itp))
    r = N5.add_ligand_posres_include(str(d/"topol.top"))
    patched["patched" if r["patched"] else "already"] += 1

devs = sorted(v["max_bond_dev_angstrom"] for v in i5.values())
print(f"I5 通过 {len(i5)}/{len(sysdirs)}")
print(f"  gro-vs-sdf 键长偏差: min {devs[0]:.4f} / median {devs[len(devs)//2]:.4f} "
      f"/ max {devs[-1]:.4f} Å   (容差 0.02；.gro 的 0.001 nm 网格量化底约 0.0095 Å)")
print(f"topol.top 补 POSRESLIG: 新增 {patched['patched']} / 已有 {patched['already']}")

# 配体残基名一致性（33 体系必须同名，否则 tc-grps / index 会逐体系不同）
resn = {N5.detect_ligand_resname(str(d/"ligand.acpype"/"ligand_GMX.itp")) for d in sysdirs}
assert len(resn) == 1, f"配体残基名不唯一: {resn}"
LIGRES = resn.pop(); print("配体残基名（33 体系一致）:", LIGRES)

I5 通过 33/33
  gro-vs-sdf 键长偏差: min 0.0069 / median 0.0099 / max 0.0143 Å   (容差 0.02；.gro 的 0.001 nm 网格量化底约 0.0095 Å)
topol.top 补 POSRESLIG: 新增 0 / 已有 33
配体残基名（33 体系一致）: MOL


In [5]:
# Cell 3 · 生成 99 条轨迹目录 + run_md2.sh（每 replica 显式种子）
import hashlib
traj, mdp_meta = [], {}
for d in sysdirs:
    for rep in REPLICAS:
        run = d/rep; run.mkdir(parents=True, exist_ok=True)
        sh, meta = N5.emit_run_md2_v3(str(run), str(d), rep, str(PR), GMXRC, PY,
                                      ligand_resname=LIGRES, md_ns=100.0,
                                      nvt_ps=100.0, npt_ps=100.0, restrain_ligand=True)
        (run/"run_md2.sh").write_text(sh)
        traj.append(str(run/"run_md2.sh"))
        mdp_meta[f"{d.name}/{rep}"] = meta

assert len(traj) == 99, len(traj)
(OUT_ROOT/"prod_scripts.txt").write_text("\n".join(traj) + "\n")

# MDP 集合在 replica 内应完全一致（除种子外），跨体系亦然 —— 断言之
by_rep = {}
for k, m in mdp_meta.items():
    by_rep.setdefault(m["replica"], set()).add(json.dumps(m["sha256"], sort_keys=True))
for rep, hs in sorted(by_rep.items()):
    assert len(hs) == 1, f"{rep} 的 MDP 在体系间不一致（{len(hs)} 种）"
print("每 replica 的 MDP 跨 33 体系唯一 ✓")
for rep in REPLICAS:
    m = mdp_meta[f"{sysdirs[0].name}/{rep}"]
    print(f"  {rep}: gen_seed={m['gen_seed']}  tc_grps={m['tc_grps']}  "
          f"帧数期望={m['n_frames_expected']}  md.mdp={m['sha256']['md.mdp']}")
print(f"\n已生成 {len(traj)} 份 run_md2.sh；清单 {OUT_ROOT/'prod_scripts.txt'}")

每 replica 的 MDP 跨 33 体系唯一 ✓
  r1: gen_seed=42  tc_grps=Protein_MOL Water_and_ions  帧数期望=10001  md.mdp=1285959a1a80813d
  r2: gen_seed=43  tc_grps=Protein_MOL Water_and_ions  帧数期望=10001  md.mdp=1285959a1a80813d
  r3: gen_seed=44  tc_grps=Protein_MOL Water_and_ions  帧数期望=10001  md.mdp=1285959a1a80813d

已生成 99 份 run_md2.sh；清单 /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/prod_scripts.txt


In [6]:
# Cell 4 · 生成生产数组（不在此提交）
(OUT_ROOT/"node05b_prod.sbatch").write_text(
    N5.emit_prod_sbatch(str(OUT_ROOT/"prod_scripts.txt"), len(traj), str(PR),
                        str(OUT_ROOT/"logs"), partition="gpu", account="hongmeilab",
                        qos="normal", gres="gpu:geforce:1", cpus=8,
                        time="14:00:00", throttle=8))
print(f"""已写 {OUT_ROOT/'node05b_prod.sbatch'}

① 先冒烟测试一条（约 8–9 h；但前 10 分钟即可判定 index / grompp / 拓扑是否成立）：
     cd {PR}
     bash {traj[0]}
   十分钟后查：
     cat {Path(traj[0]).parent}/03_min/ndx_groups.json     # 组号按名解析成功
     cat {Path(traj[0]).parent}/03_min/qc.json             # fmax / mindist / 分支
     grep -c "^" {Path(traj[0]).parent}/03_min/grompp_em.log

② 通过后提交（断点续跑：06_md/md.gro 存在即跳过）：
     cd {PR}
     sbatch {OUT_ROOT/'node05b_prod.sbatch'}

   99 条 × 100 ns，%8 并发；单条约 8.4 h（V2 实测）→ 约 13 波 ≈ 4–5 天。

③ 全部完成后回到 Cell 6 核验。
""")

已写 /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/node05b_prod.sbatch

① 先冒烟测试一条（约 8–9 h；但前 10 分钟即可判定 index / grompp / 拓扑是否成立）：
     cd /mnt/hpc/home/wanghongmei/lang2
     bash /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/pool_mol_001__6W63__primary/r1/run_md2.sh
   十分钟后查：
     cat /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/pool_mol_001__6W63__primary/r1/03_min/ndx_groups.json     # 组号按名解析成功
     cat /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/pool_mol_001__6W63__primary/r1/03_min/qc.json             # fmax / mindist / 分支
     grep -c "^" /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/pool_mol_001__6W63__primary/r1/03_min/grompp_em.log

② 通过后提交（断点续跑：06_md/md.gro 存在即跳过）：
     cd /mnt/hpc/home/wanghongmei/lang2
     sbatch /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/node05b_prod.sbatch

   99 条 × 100 ns，%8 并发；单条约 8.4 h（V2 实测）→ 约 13 波 ≈ 4–5 天。

③ 全部完成后回到 Cell 6 核验。



In [ ]:
# Cell 5 ·（可选）监控
r = N5.run("squeue -u $USER -n node05b_prod -h | wc -l")
print("排队/运行中:", r.stdout.strip())
stage = {"0 pending": 0, "1 em": 0, "2 nvt": 0, "3 npt": 0, "4 md_running": 0, "5 DONE": 0}
for s in traj:
    d = Path(s).parent
    if   (d/"06_md"/"md.gro").exists():  stage["5 DONE"] += 1
    elif (d/"06_md"/"md.log").exists():  stage["4 md_running"] += 1
    elif (d/"05_npt"/"npt.gro").exists():stage["3 npt"] += 1
    elif (d/"04_nvt").exists():          stage["2 nvt"] += 1
    elif (d/"03_min"/"em.gro").exists(): stage["1 em"] += 1
    else:                                stage["0 pending"] += 1
for k, v in sorted(stage.items()): print(f"  {k:14s} {v}")

# QC 分支分布（这是 Methods 必须报告的数字）
import collections
br = collections.Counter()
for s in traj:
    q = Path(s).parent/"03_min"/"qc.json"
    if q.exists():
        d_ = json.loads(q.read_text())
        br[(d_["equilibration"], d_["legacy_gate_branch"])] += 1
print("(平衡路径, legacy 闸判定):", dict(br))
print("  期望全部为 ('staged','repair')；出现 'main' 即需解释，勿忽略")

In [9]:
# Cell 6 · 生产后核验 + Node 05b 冻结
import pandas as pd, collections, datetime
importlib.reload(N5)

rows = []
for s in traj:
    d = Path(s).parent
    rec = {"system": d.parent.name, "rep": d.name}
    try:
        r = N5.assert_trajectory_complete(str(d), expected_frames=10001)
        rec.update({"ok": True, "seed": r["gen_seed"],
                    "equil": r["qc"]["equilibration"],
                    "legacy_gate": r["qc"]["legacy_gate_branch"],
                    "fmax": r["qc"]["fmax"], "mindist_nm": r["qc"]["mindist_nm"],
                    "n_atoms": r["n_atoms"]})

    except Exception as e:
        rec.update({"ok": False, "error": f"{type(e).__name__}: {e}"[:90]})
    rows.append(rec)

df = pd.DataFrame(rows)
npass = int(df["ok"].sum())
print(f"轨迹完成并通过核验: {npass}/{len(df)}")
if npass < len(df):
    print(df.loc[~df.ok, ["system", "rep", "error"]].to_string(index=False))
else:
    eq = N5.assert_equilibration_uniform([str(Path(s).parent) for s in traj])
    print("平衡路径统一 ✓:", eq["equilibration"], "| legacy 闸:", eq["legacy_gate"])
    if eq["legacy_gate_disagreements"]:
        print("  ⚠ legacy 闸给出 main 的轨迹（需解释）:", eq["legacy_gate_disagreements"][:5])
    print("种子分布   :", {k: sorted(set(v)) for k, v in df.groupby("rep")["seed"]})
    print("体系原子数 :", sorted(set(df.n_atoms)))
    # 每体系三条 replica 必须种子互异（真重复的前提）
    bad = [g for g, sub in df.groupby("system") if len(set(sub.seed)) != len(sub)]
    assert not bad, f"以下体系的 replica 种子不互异: {bad[:3]}"
    print("每体系 3 replica 种子互异 ✓ —— σ_PBSA 的方差有效")

    fr = {"module": "node05b_md_production", "version": N5.NODE05_VERSION,
          "created_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
          "target": TARGET, "n_trajectories": len(df),
          "protocol_source": N5.V2_PROTOCOL_SOURCE,
          "declared_deviations_from_v2": [
              "gen_seed: -1 -> explicit per-replica (42/43/44), recorded",
              "define: -DPOSRES -> -DPOSRES -DPOSRESLIG (ligand restrained in equilibration)",
              f"tc-grps ligand group: Protein_UNL -> Protein_{LIGRES}",
              "index groups resolved by name, not by hardcoded number",
              "equilibration path declared 'staged' rather than gated; V2's gate was "
              "identically true (System-SOL mindist = TIP3P O-H bond) and its main "
              "path was never executed (V2 disk: em_strong 138 / nvt.gro 0)"],
          "replica_seeds": seeds, "ligand_resname": LIGRES,
          "mdp": mdp_meta[f"{sysdirs[0].name}/r1"],
          "upstream_prep_freeze_sha256": N5.sha256_file(OUT_ROOT/"Node05_prep_freeze.json"),
          "equilibration": eq, "equilibration_mode": N5.EQUILIBRATION_MODE,
          "trajectories": rows}
    p = OUT_ROOT/"Node05b_prod_freeze.json"
    p.write_text(json.dumps(fr, indent=1, ensure_ascii=False))
    print("\n已冻结:", p, "|", N5.sha256_file(p))
    print("下一步：phase2 PBC → phase3 MM-PBSA → phase4 结构分析（先按下方清单改选择串）。")

轨迹完成并通过核验: 99/99
平衡路径统一 ✓: staged | legacy 闸: {'repair': 99}
种子分布   : {'r1': [42], 'r2': [43], 'r3': [44]}
体系原子数 : [93391, 93392, 93399, 93400, 93402, 93404, 93406, 93407, 93409, 93410, 93411, 93413, 93415, 93417, 93418, 93419, 93421, 93423, 93424, 93426, 93446, 93456]
每体系 3 replica 种子互异 ✓ —— σ_PBSA 的方差有效

已冻结: /mnt/hpc/home/wanghongmei/lang2/md_solv_v3/Node05b_prod_freeze.json | a2a7887ff5cccc4f2456048af87fbb466c8cc2ae2abdadb3cca3cfe73e35fc83
下一步：phase2 PBC → phase3 MM-PBSA → phase4 结构分析（先按下方清单改选择串）。


---

## 交接 phase2–4：三处**必须**同步的选择串

配体残基名由 `UNL`（V2）变为 `MOL`（V3）。以下三处若不改，phase3 会断言失败、
phase4 会静默算出空配体：

| 文件 | 原 | 改为 |
|---|---|---|
| `phase3_mmpbsa_solv.ipynb` | `REC_NUM, LIG_NUM = 1, 13`；`assert g[13]=='UNL'` | 按名解析：`LIG_NUM = N5.ndx_group_index(ndx, 'MOL')`，并断言 `g[LIG_NUM]=='MOL'` |
| `phase4_analysis_solv.ipynb` | `LIG = 'UNL'`；`resname UNL`（3 处） | `'MOL'` / `resname MOL` |
| `phase2_pbc_correct.ipynb` | `ARMS = ['md_solv','md_cross_solv']` | 增加 `'md_solv_v3'`（其安全断言 `endswith('_solv')` 会拒绝 `md_solv_v3`，需一并放宽为显式白名单） |

轨迹路径 `md_solv_v3/<系统>/r{1,2,3}/06_md/` 与 phase2/3/4 的 glob 深度
（`*/*/06_md/`）一致，无须改 glob。

## Methods 必须写明的三点

1. **协议来源**：MDP 参数与 V2 逐字一致，四处受控偏离已列表声明（种子、配体限制、
   配体组名、按名解析）。
2. **QC 分支**：EM 后 `fmax > 500 kJ·mol⁻¹·nm⁻¹` 或最小距离 `< 0.12 nm` 的轨迹改走
   强 EM + 分段 NVT；Cell 6 报告其条数，须在 Methods 中如实给出。
3. **分析窗口**：MM-PBSA 用 200 帧（50.01–99.76 ns）（2026-10-31 更正：原记 interval 10 / 500 帧），结构分析用 501 帧
   （50.00–99.99 ns）；同为后 50 ns，差一帧，一次说清即可。

## 尚未处理

- **跨靶点臂（3V3M / 4YLU，18 条）**：须各自的 `PROTEIN_CHARGE`、His 计数
  （7 为 6W63 专有）、`DYAD_SPEC`（MERS 为 His41/Cys148）。先跑完主臂 99 条。
- **4YLU 多链 posre**：V2 的 `run_md2.sh` 曾有该补丁；V3 受体为单链 A，届时须重新确认。
